In [ ]:
import numpy as np
import pandas as pd
import warnings

from sklearn.covariance import GraphicalLasso, GraphicalLassoCV
from sklearn.experimental import enable_iterative_imputer
from sklearn.impute import SimpleImputer, IterativeImputer
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import f1_score
from sklearn.exceptions import ConvergenceWarning

from numpy.linalg import inv, norm, eigvalsh, pinv


# Global settings

np.random.seed(123)
warnings.filterwarnings("ignore", category=ConvergenceWarning)
warnings.filterwarnings("ignore", category=RuntimeWarning)



# 1. Data Generation


def make_sparse_precision(p=50, seed=123):
    rng = np.random.default_rng(seed)
    Theta = np.zeros((p, p))

    for b in range(p // 10):
        start, end = b * 10, (b + 1) * 10
        for i in range(start, end):
            for j in range(i + 1, end):
                if abs(i - j) == 1:
                    Theta[i, j] = Theta[j, i] = 0.3

    for i in range(p):
        for j in range(i + 1, p):
            if Theta[i, j] == 0 and rng.uniform() < 0.03:
                Theta[i, j] = Theta[j, i] = rng.choice([-1, 1]) * 0.15

    for i in range(p):
        Theta[i, i] = np.sum(np.abs(Theta[i, :])) + 0.5

    # Extra safeguard
    min_eval = np.min(eigvalsh(Theta))
    if min_eval <= 0:
        Theta += np.eye(p) * (abs(min_eval) + 0.5)

    return Theta


def generate_corrupted_data(n, Theta_true, miss_rate, contam_rate, seed):
    rng = np.random.default_rng(seed)
    p = Theta_true.shape[0]
    Sigma_true = inv(Theta_true)

    X_clean = rng.multivariate_normal(np.zeros(p), Sigma_true, size=n)

    X_obs = X_clean.copy()
    contam_mask = rng.uniform(size=X_obs.shape) < contam_rate
    X_obs[contam_mask] += rng.normal(0, 8.0, size=np.sum(contam_mask))

    miss_mask = rng.uniform(size=X_obs.shape) < miss_rate
    X_obs[miss_mask] = np.nan

    return X_clean, X_obs


# 2. Utility Functions


def mask_outliers_adaptive(X):
    X_masked = X.copy()
    p = X.shape[1]

    med = np.nanmedian(X_masked, axis=0)
    mad = np.nanmedian(np.abs(X_masked - med), axis=0)
    mad = np.where(mad < 1e-6, 1e-6, mad)

    thresh = 3.0 + np.log10(p) if p > 1 else 3.5
    z = np.abs((X_masked - med) / (1.4826 * mad))
    outliers = z > thresh

    X_masked[outliers] = np.nan
    return X_masked


def winsorize_data(X, q=0.05):
    Xw = X.copy()
    for j in range(X.shape[1]):
        low = np.nanquantile(X[:, j], q)
        high = np.nanquantile(X[:, j], 1 - q)
        Xw[:, j] = np.clip(X[:, j], low, high)
    return Xw


def safe_glasso_fit(X, alpha=0.1, use_cv=False, random_state=123):
    """
    Safe GraphicalLasso / GraphicalLassoCV fit with fallback.
    Returns precision matrix on original scale.
    """
    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(X)

    # Tiny jitter for numerical stability
    rng = np.random.default_rng(random_state)
    X_scaled = X_scaled + 1e-6 * rng.normal(size=X_scaled.shape)

    try:
        if use_cv:
            model = GraphicalLassoCV(
                cv=5,
                max_iter=1000,
                tol=1e-3,
                enet_tol=1e-3
            )
        else:
            model = GraphicalLasso(
                alpha=alpha,
                max_iter=1000,
                tol=1e-3
            )

        model.fit(X_scaled)
        precision_scaled = model.precision_
        alpha_used = getattr(model, "alpha_", alpha)

    except Exception:
        try:
            # Fallback 1: stronger regularization
            alpha2 = max(alpha, 0.2)
            model = GraphicalLasso(alpha=alpha2, max_iter=2000, tol=1e-3)
            model.fit(X_scaled)
            precision_scaled = model.precision_
            alpha_used = alpha2
        except Exception:
            # Fallback 2: ridge inverse covariance
            S = np.cov(X_scaled, rowvar=False)
            S = S + 0.2 * np.eye(S.shape[0])
            precision_scaled = pinv(S)
            alpha_used = 0.2

    std_dev = np.sqrt(scaler.var_)
    std_dev = np.where(std_dev < 1e-8, 1e-8, std_dev)
    precision = precision_scaled / np.outer(std_dev, std_dev)

    return precision, alpha_used


def fit_stability_selection(X, alpha, n_subsets=10, threshold=0.8, random_state=123):
    """
    Stability selection with safe GraphicalLasso.
    """
    n, p = X.shape
    edge_counts = np.zeros((p, p))
    sub_n = int(0.8 * n)

    rng = np.random.default_rng(random_state)

    for b in range(n_subsets):
        idx = rng.choice(n, sub_n, replace=False)
        prec_sub, _ = safe_glasso_fit(X[idx], alpha=alpha, use_cv=False, random_state=random_state + b)
        edge_counts += (np.abs(prec_sub) > 1e-4).astype(int)

    stable_mask = (edge_counts / n_subsets >= threshold).astype(float)
    np.fill_diagonal(stable_mask, 1.0)
    return stable_mask


def robust_impute(X, miss_rate):
    """
    More stable imputation strategy:
    - low/moderate missing: IterativeImputer
    - high missing: median imputation
    """
    if miss_rate <= 0.2:
        imputer = IterativeImputer(max_iter=50, tol=1e-3, random_state=123)
        return imputer.fit_transform(X)
    else:
        return SimpleImputer(strategy="median").fit_transform(X)



# 3. Simulation Logic

def run_comprehensive_study(n=200, p=50, n_rep=20):
    results = []

    for miss in [0.1, 0.2, 0.3]:
        for contam in [0.05, 0.1, 0.15]:
            print(f"Processing: Missing={miss}, Contamination={contam}")

            for r in range(n_rep):
                seed = 4000 + r
                Theta_true = make_sparse_precision(p, seed=seed)
                X_clean, X_obs = generate_corrupted_data(n, Theta_true, miss, contam, seed)

               
                # Oracle
                
                t_oracle, _ = safe_glasso_fit(
                    X_clean, alpha=0.1, use_cv=False, random_state=seed
                )

                
                # Mean + Glasso
                
                X_mean = SimpleImputer(strategy="mean").fit_transform(X_obs)
                t_mean, _ = safe_glasso_fit(
                    X_mean, alpha=0.1, use_cv=False, random_state=seed + 10
                )

                
                # Winsor + Glasso
                
                X_win = winsorize_data(X_mean)
                t_win, _ = safe_glasso_fit(
                    X_win, alpha=0.1, use_cv=False, random_state=seed + 20
                )

               
                # Mask + Mean
                
                X_mom = mask_outliers_adaptive(X_obs)
                X_mom = SimpleImputer(strategy="mean").fit_transform(X_mom)
                t_mom, _ = safe_glasso_fit(
                    X_mom, alpha=0.1, use_cv=False, random_state=seed + 30
                )

               
                # Mask + CV / robust imputation
                
                X_mcv_raw = mask_outliers_adaptive(X_obs)
                X_mcv_imp = robust_impute(X_mcv_raw, miss_rate=miss)

                use_cv_flag = (miss <= 0.2)
                t_mcv, alpha_used = safe_glasso_fit(
                    X_mcv_imp,
                    alpha=0.15,
                    use_cv=use_cv_flag,
                    random_state=seed + 40
                )

                
                # Improved Robust
                
                stab_threshold = 0.8 if miss <= 0.2 else 0.6
                n_subsets = 10 if miss <= 0.2 else 6

                stable_mask = fit_stability_selection(
                    X_mcv_imp,
                    alpha=alpha_used,
                    n_subsets=n_subsets,
                    threshold=stab_threshold,
                    random_state=seed + 50
                )

                t_robust = t_mcv * stable_mask

                methods = {
                    "Oracle": t_oracle,
                    "Mean+Glasso": t_mean,
                    "Winsor+Glasso": t_win,
                    "Mask+Mean": t_mom,
                    "Mask+CV": t_mcv,
                    "Improved_Robust": t_robust
                }

                tri = np.triu_indices(p, k=1)

                for name, th in methods.items():
                    true_edges = (np.abs(Theta_true[tri]) > 1e-4).astype(int)
                    est_edges = (np.abs(th[tri]) > 1e-4).astype(int)

                    f1 = f1_score(true_edges, est_edges, zero_division=0)
                    err = norm(th - Theta_true, ord="fro")
                    n_edges = np.sum(est_edges)   # number of selected off-diagonal edges

                    results.append({
                        "missing": miss,
                        "contam": contam,
                        "method": name,
                        "F1": f1,
                        "Error": err,
                        "Edges": n_edges
                    })

    df_results = pd.DataFrame(results)

    summary = (
        df_results
        .groupby(["missing", "contam", "method"])[["F1", "Error", "Edges"]]
        .agg(["mean", "std"])
        .reset_index()
    )

    # flatten columns
    summary.columns = [
        f"{c[0]}_{c[1]}" if c[1] else c[0]
        for c in summary.columns.to_flat_index()
    ]

    return summary


if __name__ == "__main__":
    final_table = run_comprehensive_study(n_rep=20)

    print("\n--- Toy data Final Methodology Comparison (Mean & SD) ---")
    pd.set_option("display.max_columns", None)
    pd.set_option("display.width", 1200)
    print(final_table)

In [ ]:
import os

# Create folder to store figures
save_dir = "figures"
os.makedirs(save_dir, exist_ok=True)

# Generate and save plots
for miss in [0.1, 0.2, 0.3]:
    plt.figure(figsize=(4, 3))

    df_sub = final_table[final_table["missing"] == miss]

    for method in methods_to_plot:
        df_m = df_sub[df_sub["method"] == method]

        plt.errorbar(
            df_m["contam"],
            df_m["F1_mean"],
            yerr=df_m["F1_std"],
            marker="o",
            color=color_map[method],
            linestyle=linestyle_map[method],
            linewidth=2.5 if method == "Improved_Robust" else 1.8,
            markersize=6,
            capsize=3,
            alpha=1.0 if method == "Improved_Robust" else 0.7,
            label=method
        )

    #plt.title(f"Missing = {miss}", fontsize=13)
    plt.xlabel("Contamination", fontsize=13)
    plt.ylabel("F1 Score", fontsize=13)
    plt.ylim(0.05, 0.5)
    plt.tick_params(axis='both', labelsize=13)
    # Only show legend once (or you can remove entirely)
    if miss == 0.1:
        plt.legend(frameon=False, fontsize=8)

    plt.tight_layout()

    
    miss_str = str(miss).replace(".", "")
    filename = os.path.join(save_dir, f"figure_f1_miss_{miss_str}.pdf")

    plt.savefig(filename, bbox_inches="tight")

    plt.show()  

    

In [ ]:
# Real Data Study: Breast Cancer + Controlled Missing/Contamination
#   - mask_outliers_adaptive
#   - winsorize_data
#   - safe_glasso_fit
#   - fit_stability_selection
#   - robust_impute


import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.datasets import load_breast_cancer
from sklearn.impute import SimpleImputer
from sklearn.metrics import f1_score
from sklearn.preprocessing import StandardScaler


# 1. Load the real dataset


bc = load_breast_cancer()
X_real_df = pd.DataFrame(bc.data, columns=bc.feature_names)

# Standardize the original complete data once
scaler_real = StandardScaler()
X_real_clean = scaler_real.fit_transform(X_real_df.values)

print("Breast cancer data shape:", X_real_clean.shape)

# 2. Build a surrogate reference graph from the original data

# Since the true graph is unknown for real data, we use the graph
# estimated from the original unperturbed data as a surrogate reference.

Theta_ref, alpha_ref = safe_glasso_fit(
    X_real_clean,
    alpha=0.1,
    use_cv=True,
    random_state=123
)

# 3. Controlled perturbation function

def add_missing_and_contamination_real(X, miss_rate=0.1, contam_rate=0.05,
                                       contam_sd=8.0, seed=123):
    """
    Add entrywise contamination and missingness to a real dataset.
    """
    rng = np.random.default_rng(seed)
    X_obs = X.copy()

    # Add contamination
    contam_mask = rng.uniform(size=X_obs.shape) < contam_rate
    X_obs[contam_mask] += rng.normal(0, contam_sd, size=np.sum(contam_mask))

    # Add missing values
    miss_mask = rng.uniform(size=X_obs.shape) < miss_rate
    X_obs[miss_mask] = np.nan

    return X_obs

# 4. comparison study

def run_real_breast_cancer_study(
    X_clean,
    Theta_reference,
    miss_grid=[0.1, 0.2, 0.3],
    contam_grid=[0.05, 0.10, 0.15],
    n_rep=20
):
    """
    Evaluate methods on the breast cancer dataset with controlled perturbations.
    Metrics are computed against a surrogate reference graph estimated from the
    original unperturbed data.
    """
    results = []

    p = X_clean.shape[1]
    tri = np.triu_indices(p, k=1)

    # Reference support from the original data
    ref_edges = (np.abs(Theta_reference[tri]) > 1e-4).astype(int)

    for miss in miss_grid:
        for contam in contam_grid:
            print(f"Real data: Missing={miss}, Contamination={contam}")

            for r in range(n_rep):
                seed = 9000 + r

                # Controlled perturbation on real data
                X_obs = add_missing_and_contamination_real(
                    X_clean,
                    miss_rate=miss,
                    contam_rate=contam,
                    contam_sd=8.0,
                    seed=seed
                )

                
                # Mean + Glasso
                
                X_mean = SimpleImputer(strategy="mean").fit_transform(X_obs)
                t_mean, _ = safe_glasso_fit(
                    X_mean,
                    alpha=0.1,
                    use_cv=False,
                    random_state=seed + 10
                )

                
                # Winsor + Glasso
                
                X_win = winsorize_data(X_mean)
                t_win, _ = safe_glasso_fit(
                    X_win,
                    alpha=0.1,
                    use_cv=False,
                    random_state=seed + 20
                )

                
                # Mask + Mean
                
                X_mom = mask_outliers_adaptive(X_obs)
                X_mom = SimpleImputer(strategy="mean").fit_transform(X_mom)
                t_mom, _ = safe_glasso_fit(
                    X_mom,
                    alpha=0.1,
                    use_cv=False,
                    random_state=seed + 30
                )

                
                # Mask + CV
                
                X_mcv_raw = mask_outliers_adaptive(X_obs)
                X_mcv_imp = robust_impute(X_mcv_raw, miss_rate=miss)

                use_cv_flag = (miss <= 0.2)
                t_mcv, alpha_used = safe_glasso_fit(
                    X_mcv_imp,
                    alpha=0.15,
                    use_cv=use_cv_flag,
                    random_state=seed + 40
                )

               
                # Improved Robust
                
                stab_threshold = 0.8 if miss <= 0.2 else 0.6
                n_subsets = 10 if miss <= 0.2 else 6

                stable_mask = fit_stability_selection(
                    X_mcv_imp,
                    alpha=alpha_used,
                    n_subsets=n_subsets,
                    threshold=stab_threshold,
                    random_state=seed + 50
                )

                t_robust = t_mcv * stable_mask

                methods = {
                    "Mean+Glasso": t_mean,
                    "Winsor+Glasso": t_win,
                    "Mask+Mean": t_mom,
                    "Mask+CV": t_mcv,
                    "Improved_Robust": t_robust
                }

                # Compare to surrogate reference
                for name, th in methods.items():
                    est_edges = (np.abs(th[tri]) > 1e-4).astype(int)

                    f1 = f1_score(ref_edges, est_edges, zero_division=0)
                    err = norm(th - Theta_reference, ord="fro")
                    n_edges = np.sum(est_edges)

                    results.append({
                        "missing": miss,
                        "contam": contam,
                        "method": name,
                        "F1_to_ref": f1,
                        "Fro_Error_to_ref": err,
                        "Edges": n_edges
                    })

    df_results = pd.DataFrame(results)

    summary = (
        df_results
        .groupby(["missing", "contam", "method"])[["F1_to_ref", "Fro_Error_to_ref", "Edges"]]
        .agg(["mean", "std"])
        .reset_index()
    )

    summary.columns = [
        f"{c[0]}_{c[1]}" if c[1] else c[0]
        for c in summary.columns.to_flat_index()
    ]

    return df_results, summary


# 5.run it

real_results, real_summary = run_real_breast_cancer_study(
    X_clean=X_real_clean,
    Theta_reference=Theta_ref,
    miss_grid=[0.1, 0.2, 0.3],
    contam_grid=[0.05, 0.10, 0.15],
    n_rep=20
)

print("\n Real Data Comparison (Breast Cancer, Mean & SD) ")
pd.set_option("display.max_columns", None)
pd.set_option("display.width", 1200)
print(real_summary)

# 6. Optional: plot F1-to-reference trends

sns.set(style="whitegrid")

methods_to_plot = [
    "Improved_Robust",
    "Mask+CV",
    "Mask+Mean",
    "Winsor+Glasso",
    "Mean+Glasso"
]

color_map = {
    "Improved_Robust": "#d62728",
    "Mask+CV": "#1f77b4",
    "Mask+Mean": "#2ca02c",
    "Winsor+Glasso": "#ff7f0e",
    "Mean+Glasso": "#7f7f7f"
}

linestyle_map = {
    "Improved_Robust": "-",
    "Mask+CV": "--",
    "Mask+Mean": "-.",
    "Winsor+Glasso": ":",
    "Mean+Glasso": "-"
}

#save the plots:
import os

# create folder 
save_dir = "figures"
os.makedirs(save_dir, exist_ok=True)


for miss in [0.1, 0.2, 0.3]:

    plt.figure(figsize=(4, 3))

    df_sub = real_summary[real_summary["missing"] == miss]

    for method in methods_to_plot:
        df_m = df_sub[df_sub["method"] == method]

        plt.errorbar(
            df_m["contam"],
            df_m["F1_to_ref_mean"],
            yerr=df_m["F1_to_ref_std"],
            marker="o",
            color=color_map[method],
            linestyle=linestyle_map[method],
            linewidth=2.5 if method == "Improved_Robust" else 1.8,
            markersize=6,
            capsize=3,
            alpha=1.0 if method == "Improved_Robust" else 0.7,
            label=method
        )

    #plt.title(f"Missing = {miss}", fontsize=13)
    plt.xlabel("Contamination", fontsize=13)
    plt.ylabel("F1 to Reference", fontsize=13)

    plt.ylim(0.2, 0.9)  # ⭐ important for real data
    plt.xticks([0.05, 0.10, 0.15])
    plt.tick_params(axis="both", labelsize=13)

    # Only show legend once
    if miss == 0.1:
        plt.legend(frameon=False, fontsize=8)

    plt.tight_layout()

    #  save PDF
    miss_str = str(miss).replace(".", "")
    filename = os.path.join(save_dir, f"real_f1_miss_{miss_str}.pdf")

    plt.savefig(filename, bbox_inches="tight")

    plt.show()  # keep for preview

# 7. Optional: one example network from original data vs perturbed data

def precision_to_adjacency(Theta, threshold=0.05):
    """
    Convert precision matrix to adjacency matrix for visualization.
    """
    A = (np.abs(Theta) > threshold).astype(int)
    np.fill_diagonal(A, 0)
    return A

# Example: one perturbed dataset
X_obs_example = add_missing_and_contamination_real(
    X_real_clean,
    miss_rate=0.1,
    contam_rate=0.05,
    contam_sd=8.0,
    seed=999
)

X_mcv_raw = mask_outliers_adaptive(X_obs_example)
X_mcv_imp = robust_impute(X_mcv_raw, miss_rate=0.1)
Theta_mcv_ex, alpha_ex = safe_glasso_fit(
    X_mcv_imp,
    alpha=0.15,
    use_cv=True,
    random_state=1000
)

stable_mask_ex = fit_stability_selection(
    X_mcv_imp,
    alpha=alpha_ex,
    n_subsets=10,
    threshold=0.8,
    random_state=1001
)

Theta_robust_ex = Theta_mcv_ex * stable_mask_ex

A_ref = precision_to_adjacency(Theta_ref, threshold=0.05)
A_robust = precision_to_adjacency(Theta_robust_ex, threshold=0.05)

print("\nReference graph edge count:", A_ref.sum() // 2)
print("Improved_Robust graph edge count:", A_robust.sum() // 2)

In [ ]:
#Riboflavin dataset
# ============================================================
# Real Data Study: Riboflavin + Controlled Missing/Contamination
# Reuses your existing functions:
#   - mask_outliers_adaptive
#   - winsorize_data
#   - safe_glasso_fit
#   - fit_stability_selection
#   - robust_impute
# ============================================================

import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.datasets import fetch_openml
from sklearn.impute import SimpleImputer
from sklearn.metrics import f1_score
from sklearn.preprocessing import StandardScaler

# 1. Load Riboflavin dataset

def load_riboflavin_data():
    """
    Try several possible OpenML names for the Riboflavin dataset.
    If none works, raise an error and let the user provide a local file.
    """
    candidate_names = [
        "riboflavin",
        "Riboflavin",
        "RiboflavinData",
        "riboflavin_data"
    ]

    last_error = None

    for name in candidate_names:
        try:
            ds = fetch_openml(name=name, version="active", as_frame=True)
            print(f"Loaded dataset from OpenML with name: {name}")
            X_df = ds.data.copy()
            y = ds.target.copy() if ds.target is not None else None
            return X_df, y
        except Exception as e:
            last_error = e

    raise RuntimeError(
        "Could not fetch Riboflavin from OpenML. "
        "Please either:\n"
        "1) check the exact OpenML dataset name/version, or\n"
        "2) download the dataset locally and load it from a CSV file.\n"
        f"Last error: {last_error}"
    )

# 2. Optional local loader 

def load_riboflavin_from_csv(csv_path, target_col=None):
    """
    Load Riboflavin from a local CSV file.
    If target_col is given, remove it from the feature matrix.
    """
    df = pd.read_csv(csv_path)

    if target_col is not None and target_col in df.columns:
        y = df[target_col].copy()
        X_df = df.drop(columns=[target_col]).copy()
    else:
        y = None
        X_df = df.copy()

    return X_df, y

# 3. Preprocess: keep top variable genes

def preprocess_riboflavin(X_df, top_k=100):
    """
    Keep numeric columns only, remove columns with all missing values,
    and select the top_k most variable features.
    """
    X_num = X_df.select_dtypes(include=[np.number]).copy()

    # Drop columns that are entirely missing
    X_num = X_num.dropna(axis=1, how="all")

    # Median fill only for variance ranking
    X_tmp = X_num.copy()
    medians = X_tmp.median(axis=0)
    X_tmp = X_tmp.fillna(medians)

    # Select top variable genes/features
    variances = X_tmp.var(axis=0)
    top_features = variances.sort_values(ascending=False).head(top_k).index.tolist()
    X_top = X_tmp[top_features].copy()

    # Standardize
    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(X_top.values)

    return X_scaled, top_features

# 4. Controlled perturbation

def add_missing_and_contamination_real(X, miss_rate=0.1, contam_rate=0.05,
                                       contam_sd=8.0, seed=123):
    """
    Add entrywise contamination and missingness to a real dataset.
    """
    rng = np.random.default_rng(seed)
    X_obs = X.copy()

    # Add contamination
    contam_mask = rng.uniform(size=X_obs.shape) < contam_rate
    X_obs[contam_mask] += rng.normal(0, contam_sd, size=np.sum(contam_mask))

    # Add missing values
    miss_mask = rng.uniform(size=X_obs.shape) < miss_rate
    X_obs[miss_mask] = np.nan

    return X_obs

# 5. comparison study

def run_real_riboflavin_study(
    X_clean,
    Theta_reference,
    miss_grid=[0.1, 0.2, 0.3],
    contam_grid=[0.05, 0.10, 0.15],
    n_rep=20
):
    """
    Evaluate methods on Riboflavin with controlled perturbations.
    Metrics are computed against a surrogate reference graph estimated
    from the original unperturbed data.
    """
    results = []

    p = X_clean.shape[1]
    tri = np.triu_indices(p, k=1)

    ref_edges = (np.abs(Theta_reference[tri]) > 1e-4).astype(int)

    for miss in miss_grid:
        for contam in contam_grid:
            print(f"Riboflavin: Missing={miss}, Contamination={contam}")

            for r in range(n_rep):
                seed = 12000 + r

                X_obs = add_missing_and_contamination_real(
                    X_clean,
                    miss_rate=miss,
                    contam_rate=contam,
                    contam_sd=8.0,
                    seed=seed
                )

                # Mean + Glasso
                
                X_mean = SimpleImputer(strategy="mean").fit_transform(X_obs)
                t_mean, _ = safe_glasso_fit(
                    X_mean,
                    alpha=0.1,
                    use_cv=False,
                    random_state=seed + 10
                )

                
                # Winsor + Glasso
                
                X_win = winsorize_data(X_mean)
                t_win, _ = safe_glasso_fit(
                    X_win,
                    alpha=0.1,
                    use_cv=False,
                    random_state=seed + 20
                )

               
                # Mask + Mean
                
                X_mom = mask_outliers_adaptive(X_obs)
                X_mom = SimpleImputer(strategy="mean").fit_transform(X_mom)
                t_mom, _ = safe_glasso_fit(
                    X_mom,
                    alpha=0.1,
                    use_cv=False,
                    random_state=seed + 30
                )

                
                # Mask + CV
                
                X_mcv_raw = mask_outliers_adaptive(X_obs)
                X_mcv_imp = robust_impute(X_mcv_raw, miss_rate=miss)

                use_cv_flag = (miss <= 0.2)
                t_mcv, alpha_used = safe_glasso_fit(
                    X_mcv_imp,
                    alpha=0.15,
                    use_cv=use_cv_flag,
                    random_state=seed + 40
                )

                
                # Improved Robust
               
                stab_threshold = 0.8 if miss <= 0.2 else 0.6
                n_subsets = 10 if miss <= 0.2 else 6

                stable_mask = fit_stability_selection(
                    X_mcv_imp,
                    alpha=alpha_used,
                    n_subsets=n_subsets,
                    threshold=stab_threshold,
                    random_state=seed + 50
                )

                t_robust = t_mcv * stable_mask

                methods = {
                    "Improved_Robust": t_robust,
                    "Mask+CV": t_mcv,
                    "Mask+Mean": t_mom,
                    "Mean+Glasso": t_mean,
                    "Winsor+Glasso": t_win
                }

                for name, th in methods.items():
                    est_edges = (np.abs(th[tri]) > 1e-4).astype(int)

                    f1 = f1_score(ref_edges, est_edges, zero_division=0)
                    err = norm(th - Theta_reference, ord="fro")
                    n_edges = np.sum(est_edges)

                    results.append({
                        "missing": miss,
                        "contam": contam,
                        "method": name,
                        "F1_to_ref": f1,
                        "Fro_Error_to_ref": err,
                        "Edges": n_edges
                    })

    df_results = pd.DataFrame(results)

    summary = (
        df_results
        .groupby(["missing", "contam", "method"])[["F1_to_ref", "Fro_Error_to_ref", "Edges"]]
        .agg(["mean", "std"])
        .reset_index()
    )

    summary.columns = [
        f"{c[0]}_{c[1]}" if c[1] else c[0]
        for c in summary.columns.to_flat_index()
    ]

    return df_results, summary


# 6. Plot

def save_riboflavin_f1_plots(ribo_summary, save_dir="figures"):
    sns.set(style="whitegrid")
    os.makedirs(save_dir, exist_ok=True)

    methods_to_plot = [
        "Improved_Robust",
        "Mask+CV",
        "Mask+Mean",
        "Winsor+Glasso",
        "Mean+Glasso"
    ]

    color_map = {
        "Improved_Robust": "#d62728",
        "Mask+CV": "#1f77b4",
        "Mask+Mean": "#2ca02c",
        "Winsor+Glasso": "#ff7f0e",
        "Mean+Glasso": "#7f7f7f"
    }

    linestyle_map = {
        "Improved_Robust": "-",
        "Mask+CV": "--",
        "Mask+Mean": "-.",
        "Winsor+Glasso": ":",
        "Mean+Glasso": "-"
    }

    for miss in [0.1, 0.2, 0.3]:
        plt.figure(figsize=(4, 3))

        df_sub = ribo_summary[ribo_summary["missing"] == miss]

        for method in methods_to_plot:
            df_m = df_sub[df_sub["method"] == method]

            plt.errorbar(
                df_m["contam"],
                df_m["F1_to_ref_mean"],
                yerr=df_m["F1_to_ref_std"],
                marker="o",
                color=color_map[method],
                linestyle=linestyle_map[method],
                linewidth=2.5 if method == "Improved_Robust" else 1.8,
                markersize=6,
                capsize=3,
                alpha=1.0 if method == "Improved_Robust" else 0.7,
                label=method
            )

        #plt.title(f"Missing = {miss}", fontsize=13)
        plt.xlabel("Contamination", fontsize=13)
        plt.ylabel("F1 to Reference", fontsize=13)
        plt.xticks([0.05, 0.10, 0.15])
        plt.tick_params(axis="both", labelsize=13)

        # You may adjust this after seeing the actual results
        # plt.ylim(0.2, 0.9)

        if miss == 0.1:
            plt.legend(frameon=False, fontsize=8)

        plt.tight_layout()

        miss_str = str(miss).replace(".", "")
        filename = os.path.join(save_dir, f"riboflavin_f1_miss_{miss_str}.pdf")
        #plt.savefig(filename, bbox_inches="tight")
        plt.savefig(filename)
        plt.show()

# 7. Main execution

try:
    X_ribo_df, y_ribo = load_riboflavin_data()
except Exception as e:
    print(str(e))
    print("\nPlease switch to local CSV loading below if needed.")
    # Example:
    # X_ribo_df, y_ribo = load_riboflavin_from_csv("riboflavin.csv", target_col="y")
    raise

# Keep top 100 most variable genes/features
X_ribo_clean, ribo_features = preprocess_riboflavin(X_ribo_df, top_k=100)

print("Riboflavin processed shape:", X_ribo_clean.shape)

# Build surrogate reference graph from original data
Theta_ribo_ref, alpha_ribo_ref = safe_glasso_fit(
    X_ribo_clean,
    alpha=0.1,
    use_cv=True,
    random_state=321
)


ribo_results, ribo_summary = run_real_riboflavin_study(
    X_clean=X_ribo_clean,
    Theta_reference=Theta_ribo_ref,
    miss_grid=[0.1, 0.2, 0.3],
    contam_grid=[0.05, 0.10, 0.15],
    n_rep=20
)

print("\nReal Data Comparison (Riboflavin, Mean & SD)")
pd.set_option("display.max_columns", None)
pd.set_option("display.width", 1400)
print(ribo_summary)

save_riboflavin_f1_plots(ribo_summary, save_dir="figures")